In [ ]:
#!/usr/bin/env python3
"""
BayesBay + hvswdpy example: joint inversion of Rayleigh phase-velocity dispersion
and H/V spectral ratio (HVSR) using a transdimensional 1D Vs model (Voronoi1D).

Requirements:
  - bayesbay >= 0.3
  - numpy, matplotlib
  - your forward solver importable as `import hvswdpy as hv`

Notes:
  * Units: this script works in SI (m, s). Ensure your priors and forward model
    use consistent units. If you use km/s and km, convert accordingly.
  * Model: Vs(z) is unknown & piecewise-constant on a Voronoi1D tessellation.
    Vp and rho are mapped from Vs via simple empirical relations (tweak as needed).
  * Transdimensionality: the number of Voronoi sites (layers) is inferred
    between n_dimensions_min and n_dimensions_max.
  * Hierarchical noise: standard deviations for both datasets are treated as unknowns.

Author: you
"""

from __future__ import annotations
import numpy as np
import matplotlib.pyplot as plt
import bayesbay as bb
from bayesbay.discretization import Voronoi1D

# Ensure the project root (which contains `hvswdpy.py`) is importable when running from the notebook folder
import sys, os
from pathlib import Path
try:
    import hvswdpy as hv
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))
    import hvswdpy as hv



In [8]:
# --- USER INPUTS -----------------------------------------------------------------
# Observations (replace with your data):
# Frequencies for dispersion (Hz) and HV (Hz)
freq_disp = np.geomspace(0.5, 20.0, 25)   # example
freq_hv   = np.geomspace(0.5, 20.0, 40)   # example

# Observed Rayleigh phase velocities (m/s) for a chosen mode (e.g., fundamental)
# and their optional masks for valid entries (True = valid)
# Replace with your measurements
c_obs = 800 + 150 * np.exp(-freq_disp/4.0)              # fake toy data
valid_disp = np.ones_like(c_obs, dtype=bool)            # all valid here

# Observed HVSR amplitudes at freq_hv
hv_obs = 2.0 - 0.6*np.exp(-(freq_hv-1.5)**2 / 1.2)      # fake toy data

# Choose which Rayleigh mode to fit (0 = fundamental). You can also pass a list
RAY_MODE = 0

# Simple empirical Vp/Vs and rho(Vp) maps (change to what you prefer)
VP_VS = 1.77
RHO_FROM_VP = lambda vp: 1800.0 + 0.31 * vp  # kg/m^3, crude; adjust!

# Vs prior bounds (m/s) versus depth control points (m)
# Provide vectors of (vmin, vmax) at specific depths; values are interpolated in between
prior_depths = np.array([0.0, 30.0, 100.0, 300.0])
prior_vmin   = np.array([150.0, 250.0, 400.0, 700.0])
prior_vmax   = np.array([800.0, 1200.0, 1800.0, 2500.0])

# Transdimensional controls
NMIN, NMAX = 4, 18                    # allowed number of Voronoi sites (layers)
VS_PERTURB_STD = 100.0                # m/s (proposal std for Vs)
SITE_PERTURB_STD = 10.0               # m (proposal std for site positions)

# Inference controls
NCHAINS = 8
NITER = 80_000
BURN  = 20_000
SAVE_EVERY = 100

# --- IMPORT YOUR FORWARD SOLVER ---------------------------------------------------
# Expecting API compatible with your example message:
# hv.dispersion(...).rayleigh_slowness  -> array of shape (nf, nm)
# hv.hv(...) -> vector of HVSR values per frequency


# --- PRIORS ----------------------------------------------------------------------
# Depth-dependent uniform prior for Vs using control-point parameterization
vs_prior = bb.prior.UniformPrior(
    name="vs",
    vmin=prior_vmin.tolist(),
    vmax=prior_vmax.tolist(),
    position=prior_depths.tolist(),
    perturb_std=VS_PERTURB_STD,
)

# Custom initializer to encourage monotonic increase with depth (optional)
def initialize_vs(param, positions=None):
    vmin, vmax = param.get_vmin_vmax(positions)
    raw = np.random.uniform(vmin, vmax, positions.size)
    return np.maximum.accumulate(np.sort(raw))

vs_prior.set_custom_initialize(initialize_vs)

# Transdimensional 1D Voronoi for the depth axis; bounds define search domain
vor = Voronoi1D(
    name="voronoi",
    vmin=0.0,
    vmax=300.0,  # search depth (m)
    perturb_std=SITE_PERTURB_STD,
    n_dimensions=None,
    n_dimensions_min=NMIN,
    n_dimensions_max=NMAX,
    parameters=[vs_prior],
    birth_from="neighbour",
)

parameterization = bb.parameterization.Parameterization(vor)

# --- FORWARD FUNCTIONS ------------------------------------------------------------
# All forward mappers take the current state and return a data vector prediction

def forward_rayleigh(state):
    v = state["voronoi"]
    z_sites = v["discretization"]                 # Voronoi site positions (depths)
    thickness_full = Voronoi1D.compute_cell_extents(z_sites)  # length = nlayers
    vs = v["vs"]                                   # Vs at sites (m/s)
    vp = vs * VP_VS
    rho = RHO_FROM_VP(vp)

    # Fortran expects thickness array of length nlayers-1 (excluding halfspace)
    thickness = thickness_full[:-1]

    out = hv.dispersion(
        frequencies_hz=freq_disp,
        vp=vp, vs=vs, rho=rho, thickness=thickness,
        n_rayleigh_modes=max(RAY_MODE+1, 1),
        n_love_modes=0,
        precision_percent=1.0,
    )

    sl = out.rayleigh_slowness   # shape (nf, nm)
    va = out.rayleigh_valid      # shape (nf, nm) nonzero for valid

    sl_mode = sl[:, RAY_MODE]
    va_mode = (va[:, RAY_MODE] != 0)

    if not np.all(va_mode):
        idx = np.where(va_mode)[0]
        if idx.size > 0:
            sl_mode = np.interp(np.arange(sl_mode.size), idx, sl_mode[va_mode])
        else:
            sl_mode[:] = np.nanmedian(sl_mode) if np.isfinite(sl_mode).any() else 1.0/1000.0

    c_pred = 1.0 / sl_mode
    return c_pred


def forward_hvsr(state):
    v = state["voronoi"]
    z_sites = v["discretization"]
    thickness_full = Voronoi1D.compute_cell_extents(z_sites)
    vs = v["vs"]
    vp = vs * VP_VS
    rho = RHO_FROM_VP(vp)

    hv_pred, _status = hv.hv(
        frequencies_hz=freq_hv,
        vp=vp, vs=vs, rho=rho, thickness=thickness_full[:-1],
        n_rayleigh_modes=max(RAY_MODE+1, 1),
        n_love_modes=0,
        precision_percent=1.0,
        nks=0,
    )
    return hv_pred

# --- TARGETS & JOINT LIKELIHOOD --------------------------------------------------
# Hierarchical noise: let BayesBay infer std for both datasets

target_disp = bb.Target(
    name="rayleigh",
    dobs=c_obs.astype(float),
    std_min=0.5,   # m/s, tune for your data
    std_max=200.0,
    std_perturb_std=1.0,
)

target_hv = bb.Target(
    name="hvsr",
    dobs=hv_obs.astype(float),
    std_min=0.02,
    std_max=1.0,
    std_perturb_std=0.01,
)

log_likelihood = bb.LogLikelihood(
    targets=[target_disp, target_hv],
    fwd_functions=[forward_rayleigh, forward_hvsr],
)

# --- RUN INVERSION ---------------------------------------------------------------
inv = bb.BayesianInversion(
    parameterization=parameterization,
    log_likelihood=log_likelihood,
    n_chains=NCHAINS,
)

inv.run(
    sampler=None,                # defaults are fine; customize if needed
    n_iterations=NITER,
    burnin_iterations=BURN,
    save_every=SAVE_EVERY,
    verbose=True,
)

# --- POSTERIOR QUICKLOOK ---------------------------------------------------------
# Extract stacked posterior mean Vs(z) and credible intervals

def summarize_vs(chain):
    # Sample a regular depth grid and interpolate each state's Vs onto it
    z = np.linspace(0, 300, 151)
    vs_samples = []
    for state in chain.states:
        v = state["voronoi"]
        z_sites = v["discretization"]
        vs_sites = v["vs"]
        # piecewise-constant between Voronoi cell boundaries
        edges = Voronoi1D.compute_cell_boundaries(z_sites)
        zc = 0.5*(edges[:-1] + edges[1:])
        vs_interp = np.interp(z, zc, vs_sites, left=vs_sites[0], right=vs_sites[-1])
        vs_samples.append(vs_interp)
    vs_samples = np.asarray(vs_samples)
    return z, np.nanmean(vs_samples, axis=0), np.nanpercentile(vs_samples, [5,95], axis=0)

# Use the coldest chain
chain0 = inv.chains[0]
chain0.print_statistics()

z, vs_mean, (vs_p05, vs_p95) = summarize_vs(chain0)

plt.figure(figsize=(4,6))
plt.fill_betweenx(z, vs_p05, vs_p95, alpha=0.3, label='90% CI')
plt.plot(vs_mean, z, lw=2, label='Posterior mean')
plt.gca().invert_yaxis()
plt.xlabel('Vs (m/s)')
plt.ylabel('Depth (m)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


/var/folders/0f/qh8vgw8n4g570f0r6q17pdz00000gn/T/ipykernel_98763/301743560.py:139: DeprecationWarning: The 'Target' class has been moved to the 'likelihood' module. Please use 'from bayesbay.likelihood import Target' instead.
  target_disp = bb.Target(
/var/folders/0f/qh8vgw8n4g570f0r6q17pdz00000gn/T/ipykernel_98763/301743560.py:147: DeprecationWarning: The 'Target' class has been moved to the 'likelihood' module. Please use 'from bayesbay.likelihood import Target' instead.
  target_hv = bb.Target(
/var/folders/0f/qh8vgw8n4g570f0r6q17pdz00000gn/T/ipykernel_98763/301743560.py:155: DeprecationWarning: The 'LogLikelihood' class has been moved to the 'likelihood' module. Please use 'from bayesbay.likelihood import LogLikelihood' instead.
  log_likelihood = bb.LogLikelihood(


Chain ID: 3
TEMPERATURE: 1
EXPLORED MODELS: 100
ACCEPTANCE RATE: 0/100 (0.00 %)
PARTIAL ACCEPTANCE RATES:
	ParamSpacePerturbation(param_space_name=voronoi):
		BirthPerturbation(voronoi): 0.00/5.00 (0.00%)
		DeathPerturbation(voronoi): 0.00/56.00 (0.00%)
		ParamPerturbation(voronoi.discretization): 0.00/2.00 (0.00%)
		ParamPerturbation(voronoi.vs): 0.00/37.00 (0.00%)
Chain ID: 3
TEMPERATURE: 1
EXPLORED MODELS: 200
ACCEPTANCE RATE: 0/200 (0.00 %)
PARTIAL ACCEPTANCE RATES:
	ParamSpacePerturbation(param_space_name=voronoi):
		BirthPerturbation(voronoi): 0.00/10.00 (0.00%)
		DeathPerturbation(voronoi): 0.00/131.00 (0.00%)
		ParamPerturbation(voronoi.discretization): 0.00/3.00 (0.00%)
		ParamPerturbation(voronoi.vs): 0.00/56.00 (0.00%)
Chain ID: 3
TEMPERATURE: 1
EXPLORED MODELS: 300
ACCEPTANCE RATE: 0/300 (0.00 %)
PARTIAL ACCEPTANCE RATES:
	ParamSpacePerturbation(param_space_name=voronoi):
		BirthPerturbation(voronoi): 0.00/14.00 (0.00%)
		DeathPerturbation(voronoi): 0.00/204.00 (0.00%)
		Pa

RuntimeError: Chain 1 failed in perturb or forward calculation for 500 times. See above for the last exception.

In [9]:
target_hv

Target(name='hvsr, dobs=ndarray of shape (40,), is_hierarchical=True, noise_is_correlated=False, std_min=0.02, std_max=1.0, std_perturb_std=0.01)